# UI vs API — Calling LLMs the Way Production Does

**Companion notebook for V015 (Phase 3.1 — the Phase 3 opener).**

You've made your first API call already (the *APIs Explained* video). Now the real question:

> **Why does the same model give a worse answer through the API than it does in the ChatGPT UI?**

Because the chat UI is a *product* wrapped around the model. It quietly does four things for you. The API makes you do each one yourself — and that's exactly the syllabus of this video:

| What the chat UI hides | What you now control on the API |
|---|---|
| A hidden **system prompt** + silent **tools** (search, diagram rendering) | §1 — why UI ≠ API output |
| The **roles/turns** that create "memory" | §4 — message format (system / user / assistant) |
| **Token-by-token** rendering | §5 — streaming |
| **Parsing** the answer into something usable | §6 — structured output (JSON mode, tool schemas, XML) |

Same model the whole time. The only thing that changes is how much *you* do vs how much the UI did for you.

> **Run environment:** Google Colab. Keys live in **Colab Secrets** (`OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GROQ_API_KEY`). Prompts stay **inline** in the code.


## 0. Setup (Colab)

Add three keys in **Colab → 🔑 Secrets** (left sidebar), with notebook access enabled:

- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`
- `GROQ_API_KEY`

Then run the next two cells once.

> Model slugs change often. If a call 404s, update the constants in the clients cell (`OPENAI_MODEL`, `ANTHROPIC_MODEL`, `GROQ_MODEL`).


In [ ]:
# Run once
%pip install -q openai anthropic groq requests

In [ ]:
import os

from google.colab import userdata

# Pull keys from Colab Secrets into the environment (SDKs read them automatically)
for key in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY"):
    os.environ[key] = userdata.get(key)

from openai import OpenAI
from anthropic import Anthropic

# Model slugs — change here if any provider updates them
OPENAI_MODEL = "gpt-5.5"
ANTHROPIC_MODEL = "claude-sonnet-4-6"
GROQ_MODEL = "llama-3.3-70b-versatile"

openai_client = OpenAI()
anthropic_client = Anthropic()
# Groq speaks the OpenAI protocol — same SDK, different base_url (see §3)
groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=os.environ["GROQ_API_KEY"])

print("Clients ready:", OPENAI_MODEL, "|", ANTHROPIC_MODEL, "|", GROQ_MODEL)

In [ ]:
# One shared example reused across the whole notebook (continuity with the prompt-patterns video)
SAMPLE_TICKET = (
    "This is the THIRD time I'm writing. I ordered the Aurora Wireless Headphones "
    "(order #ORD-88231) on June 14th for 7,499 rupees and they arrived with a cracked "
    "left ear cup. I want a full refund, not a replacement. If I don't hear back in 48 "
    "hours I'm doing a chargeback on my card."
)
print(SAMPLE_TICKET)

## §1. The hook — same model, worse answer on the API

We run **two prompts** through the raw API (GPT-5.5). On camera, you'll have already run the *identical* prompts in the ChatGPT UI on the same model. The gap is the whole point.

**Hook A — rendering gap (mermaid).** Ask for a diagram.
- *ChatGPT UI:* draws an actual flowchart.
- *API:* hands you a raw ```mermaid code block. Nothing renders it.

**Hook B — live-info / tools gap (recent news).**
- *ChatGPT UI:* silently runs a **web search** → confident, current answer with links.
- *API:* no search tool → it has to say it can't.

> ⚠️ Note: don't use "today's date" as the contrast — modern frontier models now receive the current date through the API too. What the raw API still lacks is **live tools** (web search, code execution, rendering). That's the durable gap.

Same model both times. The UI is running tools you couldn't see.

In [ ]:
# HOOK A — the mermaid prompt, through the raw API
hook_a = (
    "Create a flowchart of how an AI agent decides which tool to call, "
    "including the loop back when a tool fails. Show it as a clear "
    "top-to-bottom diagram."
)

resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": hook_a}],
)
print(resp.choices[0].message.content)
# ^ Note: this is RAW text — a ```mermaid block. The UI would have rendered it as a picture.

In [ ]:
# HOOK B — live info, through the raw API (no web-search tool attached)
hook_b = (
    "What are the 3 biggest AI model releases from THIS week? "
    "Give the model names and a source link for each."
)

resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": hook_b}],
)
print(resp.choices[0].message.content)
# ^ The raw API has no web-search tool, so it can't fetch this. The ChatGPT UI silently
#   searches and answers. (Avoid asking the date — modern models now get that via the API.)

### The reveal

The chat UI is a **client application** sitting on top of the model. Between your keystroke and the model, it silently adds:

- a long **system prompt** (today's date, persona, "use markdown/mermaid for diagrams"),
- **tools** (web search, code interpreter, image gen),
- a **renderer** that turns markdown / LaTeX / mermaid into pretty output,
- **conversation state** (your previous turns).

The API gives you **none** of that by default. You get the raw model and exactly the tokens it produced. That's not a downgrade — it's *control*. Production runs on the API precisely because you decide every one of those pieces.

The rest of this notebook is you taking back each piece, one at a time.

---

## §2. It's just HTTP

Before any SDK: an LLM API is a plain HTTPS POST with a JSON body. The SDK is a convenience wrapper around exactly this.

In [ ]:
import requests

raw = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"},
    json={
        "model": OPENAI_MODEL,
        "messages": [{"role": "user", "content": "Say hello in one short sentence."}],
    },
)

data = raw.json()
print("HTTP", raw.status_code)
print("Reply:", data["choices"][0]["message"]["content"])
# Everything below just wraps this exact POST.

## §3. The SDKs — OpenAI, Anthropic, Groq

Three providers, the same job (handle the support ticket). Watch the **one structural difference** that trips everyone up:

- **OpenAI / Groq:** the system prompt is a message with `role: "system"` *inside* the `messages` array.
- **Anthropic:** the system prompt is a **separate top-level `system=` parameter**, and `messages` holds only user/assistant turns.

Same concept, different shape. Knowing this is half of "working across providers."

In [ ]:
# --- OpenAI: system lives INSIDE messages ---
resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise support agent. Max 3 sentences."},
        {"role": "user", "content": SAMPLE_TICKET},
    ],
)
print("OPENAI:\n", resp.choices[0].message.content)

In [ ]:
# --- Anthropic: system is a SEPARATE parameter, and you must set max_tokens ---
msg = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=300,
    system="You are a concise support agent. Max 3 sentences.",
    messages=[
        {"role": "user", "content": SAMPLE_TICKET},
    ],
)
print("ANTHROPIC:\n", msg.content[0].text)

In [ ]:
# --- Groq: it's the OpenAI SDK, just pointed at a different base_url (set in the clients cell) ---
resp = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "system", "content": "You are a concise support agent. Max 3 sentences."},
        {"role": "user", "content": SAMPLE_TICKET},
    ],
)
print("GROQ (via OpenAI SDK):\n", resp.choices[0].message.content)

### Provider cheat-sheet (screenshot this)

| | OpenAI | Anthropic | Groq |
|---|---|---|---|
| Install | `openai` | `anthropic` | `openai` (compatible) |
| Client | `OpenAI()` | `Anthropic()` | `OpenAI(base_url=…groq…)` |
| Call | `chat.completions.create` | `messages.create` | `chat.completions.create` |
| System prompt | inside `messages` | separate `system=` | inside `messages` |
| `max_tokens` | optional | **required** | optional |
| Reply text | `resp.choices[0].message.content` | `msg.content[0].text` | `resp.choices[0].message.content` |
| Why pick it | ecosystem, tools | long context, coding | speed, free tier |

**The insight:** the "OpenAI SDK" is really a *protocol* many providers implement. Learn one shape, swap a `base_url`, and most of the ecosystem opens up. Anthropic is the main one that does its own thing — and the difference is mostly *where the system prompt goes*.

## §4. Message format — the three roles ARE the memory

Remember the cliffhanger from the APIs video: *"the API has no memory."* That's because **the API is stateless** — it remembers nothing between calls. The "memory" in the ChatGPT UI is just the UI replaying the whole conversation every time.

Three roles:
- **system** — the standing instructions / persona.
- **user** — what the human said.
- **assistant** — what the model said *previously*.

To give a model "memory," you send the prior `assistant` turns back in the `messages` array yourself. That's it. That's the whole trick.

In [ ]:
# Proof 1: stateless. Ask a follow-up with NO history -> the model has no idea.
resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": "What order number did I just mention?"}],
)
print("NO HISTORY:\n", resp.choices[0].message.content, "\n")

# Proof 2: feed the prior turns back yourself -> "memory" appears.
conversation = [
    {"role": "system", "content": "You are a concise support agent."},
    {"role": "user", "content": SAMPLE_TICKET},
    {"role": "assistant", "content": "I'm sorry about the damaged headphones. I've logged your refund request for order #ORD-88231."},
    {"role": "user", "content": "What order number did I just mention?"},
]
resp = openai_client.chat.completions.create(model=OPENAI_MODEL, messages=conversation)
print("WITH HISTORY:\n", resp.choices[0].message.content)

## §5. Streaming — why the UI *feels* instant

In the UI, text appears word-by-word. That's **streaming**: the server sends tokens as they're generated instead of waiting for the whole answer. Same API, one flag (`stream=True`), and you iterate over chunks.

Two reasons it matters in production: perceived latency (the user sees movement immediately) and you can start processing before the model finishes.

In [ ]:
stream = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {"role": "system", "content": "You are a support agent."},
        {"role": "user", "content": f"Write a calm, 4-sentence reply to this ticket:\n{SAMPLE_TICKET}"},
    ],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

## §6. Structured output — getting machine-usable answers

The UI gives *humans* prose. Your *code* needs structure. Three ways to force it, from most to least guaranteed:

1. **JSON mode** — the provider guarantees valid JSON. Use this for extraction/classification (exactly what powered the prompt-patterns video).
2. **Tool-call schemas** — you describe a function with a JSON schema; the model returns arguments that fit it. This is the foundation of agents (full treatment in the tools phase — preview below).
3. **XML tags** — ask for `<tag>…</tag>` and parse it. Lightweight, works everywhere, especially natural on Anthropic.

In [ ]:
# 1) JSON mode — guaranteed valid JSON back
import json

resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    response_format={"type": "json_object"},
    messages=[
        {"role": "user", "content": (
            "Extract order_id, amount, and issue from this ticket as JSON "
            f"with exactly those keys.\n{SAMPLE_TICKET}"
        )},
    ],
)
parsed = json.loads(resp.choices[0].message.content)
print(type(parsed), parsed)

In [ ]:
# 2) Tool-call schema (PREVIEW — full agent treatment in the tools phase)
tools = [{
    "type": "function",
    "function": {
        "name": "create_refund",
        "description": "Issue a refund for an order",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "amount": {"type": "number"},
                "reason": {"type": "string"},
            },
            "required": ["order_id", "amount", "reason"],
        },
    },
}]

resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    tools=tools,
    messages=[{"role": "user", "content": SAMPLE_TICKET}],
)
call = resp.choices[0].message.tool_calls[0]
print("Model wants to call:", call.function.name)
print("With arguments:", call.function.arguments)
# Notice: the model didn't reply with prose — it returned a STRUCTURED call your code can execute.

In [ ]:
# 3) XML tags — lightweight structure, parse it yourself (great on Anthropic)
import re

msg = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=300,
    messages=[{"role": "user", "content": (
        "Summarize the ticket inside <summary></summary> tags and give the "
        "sentiment inside <sentiment></sentiment> tags.\n" + SAMPLE_TICKET
    )}],
)
text = msg.content[0].text
print(text, "\n")

summary = re.search(r"<summary>(.*?)</summary>", text, re.S)
sentiment = re.search(r"<sentiment>(.*?)</sentiment>", text, re.S)
print("Parsed summary:", summary.group(1).strip() if summary else None)
print("Parsed sentiment:", sentiment.group(1).strip() if sentiment else None)

## §7. The LangChain question (coda)

You just wrote slightly different code for each provider — different client, different system-prompt placement, different reply path. LangChain (and similar) exist to hide that behind **one interface**: write once, swap providers with a string.

```python
# illustrative — not how we'll work in this series yet
from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-5.5")   # or "anthropic:claude-sonnet-4-6"
model.invoke("Handle this ticket: ...")
```

**Why we're NOT adopting it yet:** the whole point of this video was to *see* the system prompt, the roles, the streaming, the structured output. A framework hides exactly those. You can only judge what an abstraction is doing for you once you've done it by hand — which you now have. We'll revisit frameworks later, on purpose, when the boilerplate actually starts to hurt.

> No black box: learn the raw API first, reach for the framework second.

## Recap + what's next

```
UI vs API — what the chat UI hid, and how you take it back

WHY THEY DIFFER  the UI is a product: hidden system prompt + tools + renderer + state
IT'S JUST HTTP   one POST with a JSON body; the SDK is a wrapper
SDKs             OpenAI / Groq -> system in messages | Anthropic -> system= param
MESSAGE FORMAT   system / user / assistant; API is stateless -> you replay turns = "memory"
STREAMING        stream=True -> tokens as they generate (why the UI feels instant)
STRUCTURED OUT   JSON mode (guaranteed) > tool schemas (agents) > XML tags (lightweight)
FRAMEWORKS       learn raw first; reach for LangChain only when boilerplate hurts

RULE: the UI is for humans. Production runs on the raw API, because you control
      every piece the UI hid from you.
```

**Next:** you can now *call* a model precisely — but the answer is only as good as what you put in the prompt. That's **Prompt Engineering** — the 5-block skeleton, then the 5 applied patterns (extraction, classification, transformation, generation, decomposition) we run on this same support ticket.

> Try it: re-run every cell swapping `OPENAI_MODEL` for the Groq or Anthropic client. Same concepts, your own provider.
